<h1> Example: Phonons with <span style="color: red;">QE</span>py</h1>

## Idea
QEpy wraps QE's PHonon package (`ph.x`) as `qepy.qepy_phonon_ph`. The workflow is:
 - run a normal SCF and save the ground state
 - feed an `&inputph` file to `ph.phonon()` (stdin is redirected with `set_input`)
 - read the phonon frequencies from the output

In [ ]:
# --- optional pip installs (normally leave collapsed / do not run) ---
!pip install qepy f90wrap==0.2.16
!pip install matplotlib

In [2]:
import qepy
from qepy.driver import Driver
from qepy.io import set_input, set_logfile
from qepy.io import QEInput
import qepy.qepy_phonon_ph as ph      # the ph.x wrapper
import numpy as np

## Step 1: SCF ground state for silicon

In [3]:
prefix = 'si'
outdir = './tmp/'     # keep the trailing slash!

In [4]:
scf_options = {}

scf_options['&control'] = {}
scf_options['&control']['calculation'] = 'scf'
scf_options['&control']['prefix'] = prefix
scf_options['&control']['outdir'] = outdir
scf_options['&control']['pseudo_dir'] = './data'

scf_options['&system'] = {}
scf_options['&system']['ibrav'] = 2            # FCC
scf_options['&system']['celldm(1)'] = 10.2     # Si lattice parameter (bohr)
scf_options['&system']['nat'] = 2
scf_options['&system']['ntyp'] = 1
scf_options['&system']['ecutwfc'] = 40

scf_options['&electrons'] = {}
scf_options['&electrons']['conv_thr'] = 1.0e-10

scf_options['atomic_species'] = ['Si  28.086  Si_ONCV_PBE-1.2.upf']

scf_options['atomic_positions alat'] = ['Si 0.00 0.00 0.00',
                                        'Si 0.25 0.25 0.25']

scf_options['k_points automatic'] = ['4 4 4 1 1 1']

In [5]:
driver = Driver(qe_options=scf_options, task='scf', logfile=prefix+'.scf.out')
driver.scf()
driver.save()          # ph.x restarts from this ground state
driver.stop()

## Step 2: write the `&inputph` file

In [6]:
ph_options = dict(scf_options)            # start from the SCF options
ph_options['&inputph'] = {}
ph_options['&inputph']['tr2_ph'] = 1.0e-14 
ph_options['&inputph']['prefix'] = prefix 
ph_options['&inputph']['outdir'] = outdir 
ph_options['&inputph']['fildyn'] = 'si.dyn'
ph_options['&inputph']['ldisp'] = True
ph_options['&inputph']['nq1'] = 1 
ph_options['&inputph']['nq2'] = 1 
ph_options['&inputph']['nq3'] = 1   # ldisp -> no q-point line needed
ph_options['&inputph']['trans'] = True
ph_options['&inputph']['search_sym'] = False
ph_options['&inputph']['epsil'] = False

QEInput().write_qe_input('ph.in', qe_options=ph_options, prog='pw')   # &inputph appended last

## Step 3: run `ph.x` through the wrapper

In [ ]:
set_logfile('si.ph.out')
set_input('ph.in')
ph.phonon()
set_input()

## Step 4: read the Γ-point frequencies

In [7]:
for line in open('si.ph.out'):
    if 'freq (' in line:
        print(line.rstrip())

     freq (    1) =       0.160515 [THz] =       5.354197 [cm-1]
     freq (    2) =       0.160515 [THz] =       5.354197 [cm-1]
     freq (    3) =       0.160515 [THz] =       5.354197 [cm-1]
     freq (    4) =      15.682256 [THz] =     523.103752 [cm-1]
     freq (    5) =      15.682256 [THz] =     523.103752 [cm-1]
     freq (    6) =      15.682256 [THz] =     523.103752 [cm-1]


Si shows 6 modes: 3 acoustic near 0 cm$^{-1}$ and 3 degenerate optical modes
around ~500 cm$^{-1}$ (the well-known Si zone-center optical phonon). The dynamical
matrix is also written to `si.dyn`. For a full dispersion you would run `ph.phonon()`
on a q-grid (`ldisp=.true.`), then `ph.q2r()` and `ph.matdyn()` the same way.